In [ ]:
import xarray as xr
import rioxarray as rxr
from pyproj import Transformer
import numpy as np
import glob
import re
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from utils import retrieve_url, str2time
from data_funcs import int2fstep

## Retrieve Files

In [ ]:
bands = [616, 620]
hrs = ["00", "01", "02"] # Hour of day 00-23
fstep = int2fstep(0)
doy_str = "20240101"
base_url = f"https://demo.openwfm.org/web/data/fmda/tif/{doy_str}/"

In [ ]:
for band in bands:
    for hr in hrs:
        base_filename = f"hrrr.t{hr}z.wrfprs{fstep}"
        filename = f"{base_filename}.{band}.tif"
        retrieve_url(
            f"{base_url}/{filename}",
            dest_path = f"{doy_str}/{filename}"
        )

In [ ]:
ds1 = xr.open_dataset(f"{base_filename}.616.tif")
ds2 = xr.open_dataset(f"{base_filename}.620.tif")

In [ ]:
type(ds1)

In [ ]:
ds1.band_data.shape

In [ ]:
print(ds1.band_data.dims)
print(ds1.band_data.coords)

In [ ]:
ds2.band_data.shape

In [ ]:
temp = ds1.band_data
rh = ds2.band_data

In [ ]:
type(temp)

In [ ]:
Ed = 0.924*rh**0.679 + 0.000499*np.exp(0.1*rh) + 0.18*(21.1 + 273.15 - temp)*(1 - np.exp(-0.115*rh))
Ew = 0.618*rh**0.753 + 0.000454*np.exp(0.1*rh) + 0.18*(21.1 + 273.15 - temp)*(1 - np.exp(-0.115*rh)) 

In [ ]:
type(Ed)

In [ ]:
Ed.dims

In [ ]:
plt.imshow(Ed.isel(band=0))

In [ ]:
type(Ed)

## Open Multiple Datasets

In [ ]:
file_list = [f"{doy_str}/{base_filename}.{band}.tif" for band in bands]
file_list

In [ ]:
data = xr.open_mfdataset(file_list, concat_dim="band", combine="nested")

In [ ]:
data

In [ ]:
data.band_data.shape

In [ ]:
band_df_hrrr = pd.DataFrame({
    'Band': [616, 620, 624, 628, 629, 661, 561, 612, 643],
    'hrrr_name': ['TMP', 'RH', "WIND", 'PRATE', 'APCP',
                  'DSWRF', 'SOILW', 'CNWAT', 'GFLUX'],
    'dict_name': ["temp", "rh", "wind", "rain", "precip_accum",
                 "solar", "soilm", "canopyw", "groundflux"],
    'descr': ['2m Temperature [K]', 
              '2m Relative Humidity [%]', 
              '10m Wind Speed [m/s]'
              'surface Precip. Rate [kg/m^2/s]',
              'surface Total Precipitation [kg/m^2]',
              'surface Downward Short-Wave Radiation Flux [W/m^2]',
              'surface Total Precipitation [kg/m^2]',
              '0.0m below ground Volumetric Soil Moisture Content [Fraction]',
              'Plant Canopy Surface Water [kg/m^2]',
              'surface Ground Heat Flux [W/m^2]']
})

In [ ]:
def bands_to_names(bands):
    # Get the bands from file names, assumes band_df_hrrr exists in memory
    # Find matching dict_name values in the dataframe
    dict_names = band_df_hrrr.loc[band_df_hrrr['Band'].isin(bands), 'dict_name'].tolist()
    return dict_names

In [ ]:
bands_to_names(bands)

In [ ]:
band_names = bands_to_names(bands)
data = data.assign_coords(band = ("band", band_names))

In [ ]:
data.band

In [ ]:
def calc_eqs(ds):

    # Calculate Ed based on temp and rh
    temp = ds.sel(band="temp")
    rh = ds.sel(band="rh")
    
    Ed = 0.924 * rh**0.679 + 0.000499 * np.exp(0.1 * rh) + 0.18 * (21.1 + 273.15 - temp) * (1 - np.exp(-0.115 * rh))
    Ew = 0.618 * rh**0.753 + 0.000454 * np.exp(0.1 * rh) + 0.18 * (21.1 + 273.15 - temp) * (1 - np.exp(-0.115 * rh))
    
    # Expand dims and assign new band names for Ed and Ew
    Ed = Ed.expand_dims(dim="band").assign_coords(band=["Ed"])
    Ew = Ew.expand_dims(dim="band").assign_coords(band=["Ew"])

    ds = xr.concat([ds, Ed, Ew], dim="band")
    
    return ds
    

In [ ]:
data = calc_eqs(data)

In [ ]:
data

In [ ]:
data.band

In [ ]:
features_list = ['Ed', 'Ew']

In [ ]:
subset = data.sel(band=features_list)

In [ ]:
subset.band_data.shape

In [ ]:
subset

In [ ]:
subset.dims

In [ ]:
subset.band_data.shape

In [ ]:
plt.imshow(subset.sel(band="Ed")['band_data'])

In [ ]:
file_list = ['20240101/hrrr.t00z.wrfprsf00.616.tif', '20240101/hrrr.t00z.wrfprsf00.620.tif', '20240101/hrrr.t01z.wrfprsf00.616.tif', '20240101/hrrr.t01z.wrfprsf00.620.tif', '20240101/hrrr.t02z.wrfprsf00.616.tif', '20240101/hrrr.t02z.wrfprsf00.620.tif']
file_list

In [ ]:
unique_times = sorted(set(re.search(r't\d{2}z', f).group() for f in file_list))
unique_times

In [ ]:
data.encoding['source']

In [ ]:
def extract_timestamp(file_path):
    # Extract date (parent directory) and hour from the file path
    date_str = re.search(r'(\d{8})', file_path).group(1)  # Matches YYYYMMDD
    hour_str = re.search(r't(\d{2})z', file_path).group(1)  # Matches tHHz
    
    # Combine into a datetime object
    timestamp = datetime.strptime(f"{date_str} {hour_str}", "%Y%m%d %H")
    return timestamp

In [ ]:
grouped_files = [[f for f in file_list if time in f] for time in unique_times]
grouped_files

In [ ]:
# Preprocess function to extract time information from filename and set it as a coordinate
def preprocess(ds):
    # Extract time and assign as coord
    time = extract_timestamp(ds.encoding['source'])
    ds = ds.assign_coords(time=time)  # Add time coordinate
    # Extract band name and assign as coord
    band_number = int(re.search(r'\.(\d{3})\.', ds.encoding['source']).group(1))
    band_name = bands_to_names([band_number])
    ds = ds.assign_coords(band = ("band", band_name))
    
    return ds

In [ ]:
data = xr.open_mfdataset(
    grouped_files,
    concat_dim=["time", "band"],
    combine="nested",
    preprocess=preprocess
)

In [ ]:
data

In [ ]:
data.time

In [ ]:
data.band_data.shape

In [ ]:
data.dims

In [ ]:
data.band

In [ ]:
data.coords

In [ ]:
data2 = calc_eqs(data)

In [ ]:
data2.band

In [ ]:
data2.sel(band="Ew")

## Subsetting to bbox

In [ ]:
def get_projection_info(ds, epsg = 4326):
    # Given a geotiff file (a HRRR band), 
    # return info necessary to transform lat/lon coords to the file structure
    # Inputs: 
    # ds: (osgeo.gdal.Dataset)
    # epsg: (int) default 4326 for lon/lat
    # Return: (tuple) with fields (ct, g_inv)
        # ct: (osgeo.osr.CoordinateTransformation)
        # gt_inv: (tuple) output of gdal.InvGeoTransform, also could be found with gdalinfo on command line
    gt = ds.GetGeoTransform()
    gp = ds.GetProjection()
    if(ds.RasterCount>1):
        print('Not Implemented for multiple Raster bands')
        sys.exit(-1)
    # Get Projection info
    point_srs = osr.SpatialReference()
    point_srs.ImportFromEPSG(4326) # hardcode for lon/lat
    # GDAL>=3: make sure it's x/y
    # see https://trac.osgeo.org/gdal/wiki/rfc73_proj6_wkt2_srsbarn
    point_srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
    file_srs = osr.SpatialReference()
    file_srs.ImportFromWkt(gp)
    ct = osr.CoordinateTransformation(point_srs, file_srs)
    gt_inv = gdal.InvGeoTransform(gt)

    return ct, gt_inv

In [ ]:
type(data)

In [ ]:
ds2 = rxr.open_rasterio(file_list[0])

In [ ]:
ds2.rio.transform()

In [ ]:
data.rio.crs

In [ ]:
ds2

In [ ]:
plt.imshow(ds2.isel(band=0))

In [ ]:
bbox = [37, -111, 46, -95]
crs = data.rio.crs

In [ ]:
def bbox_to_xy(bbox, crs, epsg = 4326):
    transformer = Transformer.from_crs(f"EPSG:{epsg}", crs, always_xy=True)
    # Transform the lat/lon bounding box to x/y
    minx, miny = transformer.transform(bbox[1], bbox[0])  # (min_lon, min_lat)
    maxx, maxy = transformer.transform(bbox[3], bbox[2])  # (max_lon, max_lat)

    return minx, miny, maxx, maxy

In [ ]:
minx, miny, maxx, maxy = bbox_to_xy(bbox, crs)

In [ ]:
ds2_clipped = ds2.rio.clip_box(minx=minx, miny=miny, maxx=maxx, maxy=maxy)

In [ ]:
plt.imshow(ds2_clipped.isel(band=0))

In [ ]:
ds2_clipped.coords

In [ ]:
data_clipped = data.sel(x=slice(minx, maxx), y=slice(maxy, miny))  # Note: flip y for descending order

In [ ]:
data_clipped